# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HSB-25/flyrank-ml-internship-W1-Run-the-Starter-Notebooks/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** a page is worth a CTR/metadata fix if it is genuinely visible
(real search demand, not a trickle), sits high enough in results that a click is actually
available (top 20), and still earns far fewer clicks than other pages sitting at the same
position typically do. That gap — "ranking fine, but under-clicked for where it ranks" — is
the same idea behind FlyRank's CTR-fix logic: the fix is the title/snippet/meta description,
not a full rewrite, because the ranking already works.

Before encoding it, I'm checking the two signals it leans on against the data itself — one is
the staleness idea behind FlyRank's refresh flags, the other is the CTR-vs-position idea
directly behind the CTR-fix logic that my rule uses.

In [9]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/HSB-25/flyrank-ml-internship-W1-Run-the-Starter-Notebooks"
REPO_DIR = "flyrank-ml-internship-W1-Run-the-Starter-Notebooks"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "work", "notebooks"))

print("Working dir:", os.getcwd())
assert os.path.exists("../../data/raw/content_refresh_anonymized.csv"), "CSV not found — check the repo URL/path above"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-W1-Run-the-Starter-Notebooks/work/notebooks/flyrank-ml-internship-W1-Run-the-Starter-Notebooks/work/notebooks
Starter data found. You're ready.


### Signal check A — staleness, behind FlyRank's refresh flags

**Claim:** "the longer a page goes without an update, the more likely it is declining."
This is the signal behind FlyRank's refresh flags, and the first thing most people reach for
in this lane (I did too, in ML-02/03).

**Test:** bucket every page by `freshness_tier` (days since last update) and compare decline
rate per bucket, with `n` printed so no bucket is read past its sample-size floor (~50 rows).


In [10]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Rows:", len(df), "| Clients:", df["client_id"].nunique())
print("Base decline rate (for later comparison only, not a rule input):",
      round(df["is_declining_label"].mean(), 3))

tierA = (
    df.groupby("freshness_tier")
      .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
      .round(3)
      .sort_values("decline_rate")
)
print("\nSignal check A — decline rate by freshness_tier:")
print(tierA)

df["_dslu_quintile"] = pd.qcut(df["days_since_last_update"], 5, duplicates="drop")
tierA_fine = (
    df.groupby("_dslu_quintile")
      .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
      .round(3)
)
print("\nSame check, finer quintile cut:")
print(tierA_fine)
df = df.drop(columns=["_dslu_quintile"])

Rows: 30000 | Clients: 32
Base decline rate (for later comparison only, not a rule input): 0.542

Signal check A — decline rate by freshness_tier:
                    n  decline_rate
freshness_tier                     
181+              174         0.471
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611

Same check, finer quintile cut:
                    n  decline_rate
_dslu_quintile                     
(0.999, 20.0]   15866         0.539
(20.0, 22.0]     3564         0.393
(22.0, 104.0]   10252         0.599
(104.0, 373.0]    318         0.547


/tmp/ipykernel_1511/144850573.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("_dslu_quintile")


**Verdict: MIXED.** All four `freshness_tier` buckets clear the ~50-row floor, but decline
rate does **not** rise with staleness the way the flag story predicts: `0-30` days sits at
0.511, `91-180` days is *higher* at 0.611, and `181+` days — the stalest pages — is actually
the *lowest* at 0.471. The finer quintile cut shows the same non-monotonic pattern, not a
binning artifact. Staleness alone isn't a reliable decline signal in this slice — plausibly
because the most-stale pages already declined long ago and have flattened out, so "hasn't been
touched in a year" and "actively declining right now" are different things. **This is a real,
useful negative:** it stops me from putting staleness in the score as a primary driver, which
is exactly what a hand-written rule needs to know before it ships.

### Signal check B — CTR vs. position, behind the CTR-fix logic

**Claim:** "pages ranking better get more clicks per impression" — i.e. CTR should fall as
position gets worse. This is the signal the CTR-fix flag leans on: a page underperforming the
CTR that other pages at its *own* position typically earn is the definition of a fixable gap.

**Test:** apply a volume floor first (`impressions_90d >= 100`, the data dictionary's warning
about reading position tiers without a floor), then bucket by `position_tier` and compare
median/mean CTR, with `n` printed per bucket.

In [11]:
floor = df[df["impressions_90d"] >= 100].copy()
print(f"Rows before floor: {len(df):,} | after impressions_90d >= 100 floor: {len(floor):,}")
print()

order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
tierB = (
    floor[floor["position_tier"].isin(order)]
    .groupby("position_tier")
    .agg(n=("content_id", "size"),
         median_ctr=("ctr", "median"),
         mean_ctr=("ctr", "mean"),
         decline_rate=("is_declining_label", "mean"))
    .round(3)
    .reindex(order)
)
print(tierB)

Rows before floor: 30,000 | after impressions_90d >= 100 floor: 22,006

                  n  median_ctr  mean_ctr  decline_rate
position_tier                                          
top_3           533        0.19     0.334         0.756
page_1         8633        0.23     0.355         0.607
striking       5903        0.15     0.256         0.626
page_3_5       6058        0.06     0.142         0.584
deep            879        0.00     0.055         0.317


**Verdict: CONFIRMED.** Every bucket clears the floor by a wide margin (n = 533 to 8,633).
CTR falls in a clear step pattern from `page_1` down to `deep` (mean CTR 0.355 → 0.256 → 0.142
→ 0.055) — worse position really does mean fewer clicks per impression, which is what the
CTR-fix flag assumes. One honest nuance: `top_3` (n=533, mean CTR 0.334) sits slightly *below*
`page_1` (mean CTR 0.355) instead of above it — likely because `top_3` is thin (the data
dictionary's warning about low-volume top positions) and pulled around by a few outliers even
after the floor. The overall position → CTR relationship holds; I won't over-read the exact
ordering of the top two buckets against each other.

### The rule, encoded

Putting the two verdicts together: staleness is out (MIXED, not safe to lean on); the
CTR-vs-position gap is in (CONFIRMED) — combined with a plain visibility floor so the rule
never chases pages nobody sees.

- **Eligibility (gate):** `impressions_90d >= 100` (real, measurable demand) **and**
  `avg_position` is known and `<= 20` (page 1 or 2 — a click is actually reachable) **and**
  the page's CTR is below the *median CTR other pages earn at its own position tier*
  (the CTR gap must be positive).
- **Score:** `ctr_gap × log1p(impressions_90d)`, zero for anything that fails the gate — a
  bigger gap on a bigger audience ranks first. No fitted weights, fully readable.
- **One reason code:** `low_ctr_for_position` when the gate passes, else `not_flagged`.
- **One action label:** `fix_ctr_and_metadata` when flagged, else `monitor`.

`content_type`, `content_age_days`, `word_count`, and every product tier column
(`impression_tier`, `freshness_tier`, etc.) are deliberately **not** in the score — they're
either label-adjacent or exactly the kind of pre-computed product flag the flyrank-data skill
says can be a baseline to beat but never a rule input.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
import os

benchmark_ctr = (
    floor[floor["position_tier"].isin(order)]
    .groupby("position_tier")["ctr"]
    .median()
)
print("Benchmark (median) CTR per position tier, floor >= 100 impressions:")
print(benchmark_ctr.round(3))
print()

df["benchmark_ctr"] = df["position_tier"].map(benchmark_ctr)
df["ctr_gap"] = (df["benchmark_ctr"] - df["ctr"]).clip(lower=0)

eligible = (
    (df["impressions_90d"] >= 100)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr_gap"] > 0)
)
df["eligible"] = eligible.astype(int)

df["baseline_action_score"] = df["eligible"] * df["ctr_gap"] * np.log1p(df["impressions_90d"])
df["reason_code"] = np.where(df["eligible"] == 1, "low_ctr_for_position", "not_flagged")
df["suggested_action"] = np.where(df["eligible"] == 1, "fix_ctr_and_metadata", "monitor")
df["rank"] = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)

print("Flagged rows:", int(df['eligible'].sum()), "of", len(df),
      f"({df['eligible'].mean():.1%})")

def precision_at_k(scores, labels, k):
    order_idx = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order_idx[:k]].mean()

base_rate = df["is_declining_label"].mean()
print(f"\nBase decline rate: {base_rate:.3f}")
for k in (20, 50, 100):
    p = precision_at_k(df["baseline_action_score"].values, df["is_declining_label"].values, k)
    print(f"precision@{k}: {p:.3f}  (base rate {base_rate:.3f}, lift x{p / base_rate:.2f})")

output_cols = [
    "content_id", "client_id", "rank", "baseline_action_score", "reason_code",
    "suggested_action", "avg_position", "position_tier", "ctr", "benchmark_ctr",
    "ctr_gap", "impressions_90d", "content_type", "main_intent", "trend_direction",
    "is_declining_label",
]

out = df[output_cols].sort_values("rank").reset_index(drop=True)

os.makedirs("../outputs", exist_ok=True)
out_path = "../outputs/baseline_action_score.csv"
out.to_csv(out_path, index=False)
print(f"\nWrote {len(out):,} ranked rows to {out_path}")
out.head(10)

Benchmark (median) CTR per position tier, floor >= 100 impressions:
position_tier
deep        0.00
page_1      0.23
page_3_5    0.06
striking    0.15
top_3       0.19
Name: ctr, dtype: float64

Flagged rows: 7400 of 30000 (24.7%)

Base decline rate: 0.542
precision@20: 0.700  (base rate 0.542, lift x1.29)
precision@50: 0.680  (base rate 0.542, lift x1.25)
precision@100: 0.680  (base rate 0.542, lift x1.25)

Wrote 30,000 ranked rows to ../outputs/baseline_action_score.csv


,content_id,client_id,rank,baseline_action_score,reason_code,suggested_action,avg_position,position_tier,ctr,benchmark_ctr,ctr_gap,impressions_90d,content_type,main_intent,trend_direction,is_declining_label
0,content_c8e9d6ab9013,client_19581e27de,1,2.817167,low_ctr_for_position,fix_ctr_and_metadata,9.7,page_1,0.00,0.23,0.23,208678,keyword article,informational,down,1
1,content_453722754fea,client_f369cb89fc,2,2.606993,low_ctr_for_position,fix_ctr_and_metadata,7.6,page_1,0.01,0.23,0.22,140079,keyword article,informational,down,1
2,content_39881853ef0c,client_f369cb89fc,3,2.558629,low_ctr_for_position,fix_ctr_and_metadata,7.2,page_1,0.01,0.23,0.22,112434,keyword article,informational,down,1
3,content_c84a0ab98e90,client_f369cb89fc,4,2.463229,low_ctr_for_position,fix_ctr_and_metadata,7.8,page_1,0.03,0.23,0.20,223271,keyword article,informational,stable,0
4,content_0919dd345d80,client_4e07408562,5,2.454629,low_ctr_for_position,fix_ctr_and_metadata,7.0,page_1,0.02,0.23,0.21,119217,keyword article,informational,down,1
5,content_d274ac4158ef,client_4e07408562,6,2.438541,low_ctr_for_position,fix_ctr_and_metadata,6.8,page_1,0.01,0.23,0.22,65138,keyword article,informational,stable,0
6,content_e5f459e737b7,client_f369cb89fc,7,2.406709,low_ctr_for_position,fix_ctr_and_metadata,5.9,page_1,0.01,0.23,0.22,56363,keyword article,transactional,down,1
7,content_339b357d04c7,client_bbb965ab0c,8,2.366176,low_ctr_for_position,fix_ctr_and_metadata,3.7,page_1,0.01,0.23,0.22,46879,keyword article,informational,up,0
8,content_c1fe78bc4e37,client_19581e27de,9,2.361203,low_ctr_for_position,fix_ctr_and_metadata,7.5,page_1,0.03,0.23,0.20,134055,keyword article,commercial,down,1
9,content_65114d89496d,client_19581e27de,10,2.350564,low_ctr_for_position,fix_ctr_and_metadata,6.5,page_1,0.02,0.23,0.21,72631,keyword article,transactional,down,1


Also saving the run's metrics as a small JSON receipt (`work/outputs/*.json` — the file
type the assignment says IS worth committing, unlike the data CSV).

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


In [13]:
top20 = out.head(20).copy()
display_cols = ["rank", "content_id", "avg_position", "ctr", "benchmark_ctr",
                 "impressions_90d", "content_type", "main_intent", "trend_direction",
                 "is_declining_label"]
top20[display_cols]


,rank,content_id,avg_position,ctr,benchmark_ctr,impressions_90d,content_type,main_intent,trend_direction,is_declining_label
0,1,content_c8e9d6ab9013,9.7,0.00,0.23,208678,keyword article,informational,down,1
1,2,content_453722754fea,7.6,0.01,0.23,140079,keyword article,informational,down,1
2,3,content_39881853ef0c,7.2,0.01,0.23,112434,keyword article,informational,down,1
3,4,content_c84a0ab98e90,7.8,0.03,0.23,223271,keyword article,informational,stable,0
4,5,content_0919dd345d80,7.0,0.02,0.23,119217,keyword article,informational,down,1
5,6,content_d274ac4158ef,6.8,0.01,0.23,65138,keyword article,informational,stable,0
6,7,content_e5f459e737b7,5.9,0.01,0.23,56363,keyword article,transactional,down,1
7,8,content_339b357d04c7,3.7,0.01,0.23,46879,keyword article,informational,up,0
8,9,content_c1fe78bc4e37,7.5,0.03,0.23,134055,keyword article,commercial,down,1
9,10,content_65114d89496d,6.5,0.02,0.23,72631,keyword article,transactional,down,1


All 20 are `keyword article` pages sitting mid-page-1 (avg position ~4–10) with CTR at or
near 0.00–0.05% against a ~0.23% benchmark for that position — huge audiences (16k–295k
impressions/90d) barely clicking. One line each: **action — why it's there — what would make
it wrong.**

1. **fix_ctr_and_metadata** — page 1 (9.7), CTR 0.00 vs 0.23 benchmark, 208k impressions, and
   already labeled `down` — the single biggest gap on the biggest audience. *Wrong if:* the
   query is already highly branded/navigational, where low CTR is normal, not a title problem.
2. **fix_ctr_and_metadata** — page 1 (7.6), CTR 0.01, 140k impressions, `down` — same pattern,
   second-largest audience. *Wrong if:* a SERP feature (featured snippet, PAA) above this
   result is eating clicks that no title rewrite can win back.
3. **fix_ctr_and_metadata** — page 1 (7.2), CTR 0.01, 112k impressions, `down`. *Wrong if:*
   `avg_position` is a 90-day average masking a page that just fell off page 1 recently — the
   real fix would be re-earning rank, not the snippet.
4. **fix_ctr_and_metadata** — page 1 (7.8), CTR 0.03, 223k impressions, but trend is `stable`,
   not `down` — largest audience in the whole top 20. *Wrong if:* this CTR is simply normal for
   this query's intent and always has been; nothing here is actually changing.
5. **fix_ctr_and_metadata** — page 1 (7.0), CTR 0.02, 119k impressions, `down`. *Wrong if:* the
   page ranks for a very broad head-term where low CTR is structural (informational query,
   answer shown on the SERP itself).
6. **fix_ctr_and_metadata** — page 1 (6.8), CTR 0.01, 65k impressions, but `stable`. *Wrong if:*
   CTR has been flat at this level for a year — "underperforming its tier" and "actually
   fixable" are not the same claim.
7. **fix_ctr_and_metadata** — page 1 (5.9), CTR 0.01, 56k impressions, `down`, transactional
   intent — a transactional query with near-zero CTR is a strong genuine candidate. *Wrong if:*
   the title/meta already accurately describes a page that visitors correctly skip (price or
   fit mismatch, not a snippet problem).
8. **fix_ctr_and_metadata** — page 1 (3.7), CTR 0.01, 46k impressions, but trend is `up` —
   flagged despite already improving. *Wrong if:* whatever's driving the "up" trend is about to
   lift CTR too; touching the metadata now could interrupt something already working.
9. **fix_ctr_and_metadata** — page 1 (7.5), CTR 0.03, 134k impressions, `down`, commercial
   intent. *Wrong if:* this is a comparison-style query where users click straight to a
   different SERP feature (shopping carousel) regardless of title.
10. **fix_ctr_and_metadata** — page 1 (6.5), CTR 0.02, 72k impressions, `down`, transactional.
    *Wrong if:* the low CTR reflects a genuinely worse product fit, and a better title would
    only raise bounce rate, not real conversions.
11. **fix_ctr_and_metadata** — page 1 (8.0), CTR 0.03, 123k impressions, but trend is `up` —
    same caution as #8: flagging an already-improving page. *Wrong if:* the improvement is
    early and the metadata isn't actually the bottleneck yet.
12. **fix_ctr_and_metadata** — page 1 (7.8), CTR 0.02, 65k impressions, `down`. *Wrong if:*
    `avg_position` here blends two very different queries this page ranks for, and the real
    average position for its main query is worse than 20 (gate wouldn't have caught that).
13. **fix_ctr_and_metadata** — page 1 (9.1), CTR 0.01, 38k impressions, `down`. *Wrong if:* this
    page sits at the bottom edge of page 1 and mobile users never scroll to see it — a ranking
    problem, not a snippet problem.
14. **fix_ctr_and_metadata** — page 1 (6.6), CTR 0.00, 22k impressions, `down`, intent missing
    (`main_intent` is blank). *Wrong if:* the missing intent means this page was never properly
    classified, so I can't even confirm the query type this "CTR benchmark" should apply to.
15. **fix_ctr_and_metadata** — page 1 (6.4), CTR 0.03, 99k impressions, `down`. *Wrong if:*
    same-page cannibalization — another page from this client already ranks above it for the
    same query and is absorbing the clicks.
16. **fix_ctr_and_metadata** — page 1 (7.3), CTR 0.05, 295k impressions — the single largest
    audience in the queue, but trend is `stable`. *Wrong if:* at this scale, 0.05% CTR is
    already the realistic ceiling for a very broad, low-intent head-term.
17. **fix_ctr_and_metadata** — page 1 (6.6), CTR 0.03, 83k impressions, `down`, transactional.
    *Wrong if:* seasonality — a transactional query with a strong seasonal pattern would show
    "low CTR" right now for timing reasons the rule can't see.
18. **fix_ctr_and_metadata** — page 1 (9.0), CTR 0.02, 44k impressions, but trend is `stable`.
    *Wrong if:* this is a duplicate/near-duplicate of another client page and splits clicks
    with it — the real problem is content overlap, not the title.
19. **fix_ctr_and_metadata** — page 1 (5.6), CTR 0.00, 16k impressions, `down`. *Wrong if:* the
    page's actual rendered title/meta already differ from what's stored in this export — I'm
    scoring metadata I haven't actually seen.
20. **fix_ctr_and_metadata** — page 1 (9.0), CTR 0.00, 16k impressions, `down`. *Wrong if:*
    smallest audience in the top 20 — a couple of weeks of zero clicks could just be noise at
    this volume, not a real pattern yet.

In [14]:
import json

metrics = {
    "n_rows": int(len(df)),
    "n_flagged": int(df["eligible"].sum()),
    "flagged_share": float(df["eligible"].mean()),
    "base_decline_rate": float(base_rate),
    "precision_at_20": float(precision_at_k(df["baseline_action_score"].values,
                                             df["is_declining_label"].values, 20)),
    "precision_at_50": float(precision_at_k(df["baseline_action_score"].values,
                                             df["is_declining_label"].values, 50)),
    "precision_at_100": float(precision_at_k(df["baseline_action_score"].values,
                                              df["is_declining_label"].values, 100)),
    "signal_verdicts": {
        "staleness_freshness_tier": "MIXED",
        "ctr_vs_position_tier": "CONFIRMED",
    },
    "rule": {
        "gate": "impressions_90d >= 100 AND 0 < avg_position <= 20 AND ctr_gap > 0",
        "score_formula": "eligible * ctr_gap * log1p(impressions_90d)",
        "reason_code": "low_ctr_for_position (else not_flagged)",
        "action": "fix_ctr_and_metadata (else monitor)",
    },
}

with open("../outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))


{
  "n_rows": 30000,
  "n_flagged": 7400,
  "flagged_share": 0.24666666666666667,
  "base_decline_rate": 0.5420666666666667,
  "precision_at_20": 0.7,
  "precision_at_50": 0.68,
  "precision_at_100": 0.68,
  "signal_verdicts": {
    "staleness_freshness_tier": "MIXED",
    "ctr_vs_position_tier": "CONFIRMED"
  },
  "rule": {
    "gate": "impressions_90d >= 100 AND 0 < avg_position <= 20 AND ctr_gap > 0",
    "score_formula": "eligible * ctr_gap * log1p(impressions_90d)",
    "reason_code": "low_ctr_for_position (else not_flagged)",
    "action": "fix_ctr_and_metadata (else monitor)"
  }
}


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [15]:
weak = top20[top20["trend_direction"] != "down"]
print("Weak picks in the top 20 (flagged, but NOT labeled 'down'):", len(weak))
weak[["rank", "content_id", "avg_position", "ctr", "impressions_90d", "trend_direction",
      "is_declining_label"]]


Weak picks in the top 20 (flagged, but NOT labeled 'down'): 6


,rank,content_id,avg_position,ctr,impressions_90d,trend_direction,is_declining_label
3,4,content_c84a0ab98e90,7.8,0.03,223271,stable,0
5,6,content_d274ac4158ef,6.8,0.01,65138,stable,0
7,8,content_339b357d04c7,3.7,0.01,46879,up,0
10,11,content_b115f7c74779,8.0,0.03,123469,up,0
15,16,content_36ff89c8214e,7.3,0.05,295097,stable,0
17,18,content_f6ae0f36d70d,9.0,0.02,44860,stable,0


**Weak picks: ranks 4, 6, 8, 11, 16, 18 — six of the top 20 (30%).** Four are `stable`
(#4, #6, #16, #18) and two are actually `up` (#8, #11). All six share the same story: they're
huge-audience, low-CTR pages that clear the eligibility gate on volume and position alone — the
score's `log1p(impressions_90d)` term can out-weigh a modest CTR gap and pull a page that isn't
declining to the top just because its audience is enormous. That's a real weakness of a
two-factor score: it finds "under-clicked for its position," which is true for all six, but
that's not the same claim as "declining," and a content team acting on this queue should read
`trend_direction` alongside the score before treating every top-20 row as urgent.

**Leakage check — confirmed clean:**
- **No label-derived inputs.** `trend_direction` and `trend_pct` (the label source) are read
  only for the review above and the precision@K check, never fed into `eligible`,
  `ctr_gap`, or `baseline_action_score`.
- **No product flags as inputs.** `impression_tier`, `freshness_tier`, `age_tier`,
  `position_tier` labels aren't used as score inputs either — `position_tier` is only used to
  compute the benchmark CTR lookup (a value, not a flag decision), and the actual gate re-checks
  `avg_position` directly as a number.
- **No future window.** Everything here comes from the trailing-90-day snapshot
  (`impressions_90d`, `avg_position`, `ctr`); `impressions_last_30d` / `impressions_prev_30d`
  (the trend inputs) never appear in the score.
- **No client names, URLs, or private queries** anywhere in this notebook or the output CSV —
  only pseudonymous `content_id` / `client_id`.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.